# gubapost 特征提取 → stock-day mean-pooled 特征

**一次前向, 两个产物, 后续全部 CPU 不回 GPU。**

| 产物 | 内容 | 大小 |
|---|---|---|
| `gubapost_stockday_cls.parquet` | CLS[768] + class_1_prob + n_posts | ~1.5GB |
| `gubapost_stockday_attn.parquet` | 12 层 dead 128 投影 [12×128] + n_posts | ~3-15GB |

**加速**: SDPA + 双卡 + 长度分桶 + 沪深 A 股过滤
**Hooks**: 12 个 `attention.output.dense` read-only, hook 内直接投影到 dead 128 (不存全量 768)
**注意**: 设 `YEARS` 可先跑部分年份

In [ ]:
# 1. 配置
import os, sys
ROOT = "/home/intern_fjq_2026/Projects/chinese-wwm-roberta"
os.chdir(ROOT); sys.path.insert(0, ROOT)
INPUT_DIR = "/home/intern_fjq_2026/data/NLP/gubapost"
OUT_DIR = os.path.join(ROOT, "artifacts", "gubapost_cls")
SCRIPT = os.path.join(ROOT, "scripts", "extract_gubapost_cls.py")

# ★ 改这里控制跑哪些年份: None=全部, "2020,2021"=先跑两年
YEARS = "2020,2021,2022,2023,2024,2025,2026"

GPUS = "0,1"; WORKERS = 8; TOKEN_WORKERS = 4
BATCH_TOKENS = 131072; BATCH_ROWS = 4096; MAX_LENGTH = 512; DTYPE = "fp16"
os.makedirs(OUT_DIR, exist_ok=True)
import torch
print("CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    f, t = torch.cuda.mem_get_info(i); print(f"  GPU{i}: 空闲 {f/2**30:.0f}G/{t/2**30:.0f}G")
print(f"配置: years={YEARS}, gpus={GPUS}, workers={WORKERS}, max_length={MAX_LENGTH}")

CUDA: True | GPUs: 2
  GPU0: 空闲 78G/79G
  GPU1: 空闲 78G/79G
配置: years=2020,2021,2023,2024,2025,2026, gpus=0,1, workers=8, max_length=512


In [2]:
# 2. 文件清单 + ETA (按 YEARS 过滤)
import glob, re
from concurrent.futures import ProcessPoolExecutor
import pyarrow.parquet as pq, pandas as pd

YEARS_SET = set(YEARS.split(",")) if YEARS else None

def n_rows(p):
    try: return p, pq.ParquetFile(p).metadata.num_rows
    except: return p, -1
files = []
for y in sorted(os.listdir(INPUT_DIR)):
    if YEARS_SET and y not in YEARS_SET: continue
    d = os.path.join(INPUT_DIR, y)
    if os.path.isdir(d): files += sorted(glob.glob(os.path.join(d, "gubapost*.parquet")))
files = [f for f in files if re.fullmatch(r"gubapost\d{8}\.parquet", os.path.basename(f))]
assert files, "没有找到文件"
with ProcessPoolExecutor(min(16, os.cpu_count())) as ex:
    inv = pd.DataFrame([{"n": n, "year": re.search(r"/(\d{4})/", p).group(1)}
        for p, n in ex.map(n_rows, files) if n >= 0])
TOTAL = int(inv.n.sum()); SHSZ = int(TOTAL * 0.983)
print(f"日文件: {len(inv)} ({inv.year.min()}~{inv.year.max()}) | 总帖: {TOTAL:,} | 沪深: ~{SHSZ:,}")
print(inv.groupby("year").n.agg(["sum","count"]).to_string())
n_gpu = len(GPUS.split(","))
print(f"预计: {n_gpu}卡+SDPA 约 {SHSZ/50000/3600:.1f}h")

日文件: 2059 (2020~2026) | 总帖: 285,252,178 | 沪深: ~280,402,890
           sum  count
year                 
2020  45467374    366
2021  58746733    365
2023  40826226    365
2024  39618216    365
2025  54197330    361
2026  46396299    237
预计: 2卡+SDPA 约 1.6h


In [3]:
# 3. 启动 (幂等)
import subprocess, json
PID_FILE = os.path.join(OUT_DIR, "run.pid")
def _running():
    if not os.path.exists(PID_FILE): return None
    try:
        pid = int(open(PID_FILE).read().strip())
        with open(f"/proc/{pid}/cmdline") as f: return pid if "extract_gubapost" in f.read() else None
    except: return None
skip = False
mf = os.path.join(OUT_DIR, "manifest.json")
if os.path.exists(mf):
    m = json.load(open(mf))
    if m.get("schema") == "gubapost_stockday_features_v1":
        print(f"已完成: {m.get('n_stock_days',0):,} stock-days"); skip = True
if not skip:
    pid = _running()
    if pid: print(f"运行中: pid={pid}, 跳过")
    else:
        cmd = [sys.executable, SCRIPT, "--input-dir", INPUT_DIR, "--out-dir", OUT_DIR,
               "--gpus", GPUS, "--workers", str(WORKERS), "--token-workers", str(TOKEN_WORKERS),
               "--batch-tokens", str(BATCH_TOKENS), "--batch-rows", str(BATCH_ROWS),
               "--max-length", str(MAX_LENGTH), "--dtype", DTYPE]
        if YEARS:
            cmd += ["--years", YEARS]
        logf = open(os.path.join(OUT_DIR, "run.log"), "a")
        p = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, cwd=ROOT,
                             env=os.environ.copy(), start_new_session=True)
        with open(PID_FILE, "w") as f: f.write(str(p.pid))
        print(f"已启动: pid={p.pid} (years={YEARS}, gpus={GPUS}, {WORKERS} workers)")

已启动: pid=20431 (years=2020,2021,2023,2024,2025,2026, gpus=0,1, 8 workers)


In [ ]:
# 4. (可选)停止
import os, signal
pf = os.path.join(OUT_DIR, "run.pid")
if os.path.exists(pf):
    os.kill(int(open(pf).read().strip()), signal.SIGTERM); os.remove(pf); print("已停止")
else: print("无运行中进程")

In [25]:
# 5. 进度监控 (反复执行)
import glob, re, os, time
pf = os.path.join(OUT_DIR, "run.pid")
if os.path.exists(pf):
    try: print(f"运行中: pid={int(open(pf).read().strip())}")
    except: pass
done = 0; rows = 0
for lf in sorted(glob.glob(os.path.join(OUT_DIR, "logs", "worker*.log"))):
    for line in open(lf):
        if "rows=" in line:
            done += 1; m = re.search(r"rows=(\d+)", line)
            if m: rows += int(m.group(1))
print(f"完成: {done}/{len(files)} 文件, ~{rows:,} 帖 ({rows/max(TOTAL,1)*100:.1f}%)")
if done and os.path.exists(pf):
    rate = rows / max(time.time() - os.path.getmtime(pf), 1)
    print(f"均速: {rate:,.0f} 帖/s, 剩余约 {(TOTAL-rows)/max(rate,1)/3600:.1f}h")
for lf in sorted(glob.glob(os.path.join(OUT_DIR, "logs", "worker*.log"))):
    ls = open(lf).read().strip().splitlines()
    print(f"--- {os.path.basename(lf)} ---")
    for l in ls[-2:]: print("  ", l[:150])
os.system("nvidia-smi --query-gpu=index,memory.used,utilization.gpu --format=csv,noheader")

运行中: pid=20431
完成: 2059/2059 文件, ~281,075,564 帖 (98.5%)
均速: 92,462 帖/s, 剩余约 0.0h
--- worker0.log ---
     return self.fget.__get__(instance, owner)()
   [09-05 11:06:58] worker0 结束: done=0 skip=258 fail=0
--- worker1.log ---
   [09-05 11:26:12] (258/258) gubapost20210530.parquet rows=44093 stocks=3900 14.18s
   [09-05 11:26:13] worker1 结束: done=25 skip=233 fail=0
--- worker2.log ---
   [09-05 11:43:33] (258/258) gubapost20230212.parquet rows=28484 stocks=4408 11.15s
   [09-05 11:43:34] worker2 结束: done=61 skip=197 fail=0
--- worker3.log ---
     return self.fget.__get__(instance, owner)()
   [09-05 11:06:59] worker3 结束: done=0 skip=258 fail=0
--- worker4.log ---
     return self.fget.__get__(instance, owner)()
   [09-05 11:06:59] worker4 结束: done=0 skip=258 fail=0
--- worker5.log ---
     return self.fget.__get__(instance, owner)()
   [09-05 11:06:59] worker5 结束: done=0 skip=258 fail=0
--- worker6.log ---
   [09-05 11:38:28] (258/258) gubapost20251215.parquet rows=232217 stocks=9610 55

0

In [24]:
# 5b. 实时进度条
import glob, re, os, threading, time
import ipywidgets as W
from IPython.display import display
if "_feat_stop" in globals(): _feat_stop.set()
_feat_stop = threading.Event()
_t0 = os.path.getmtime(os.path.join(OUT_DIR, "run.pid")) if os.path.exists(os.path.join(OUT_DIR, "run.pid")) else None
bar = W.IntProgress(min=0, max=TOTAL, description="帖子:"); label = W.HTML(); display(W.VBox([bar, label]))
def _w():
    while not _feat_stop.is_set():
        rows = 0; files_done = 0
        for lf in glob.glob(os.path.join(OUT_DIR, "logs", "worker*.log")):
            for line in open(lf):
                if "rows=" in line:
                    files_done += 1; m = re.search(r"rows=(\d+)", line)
                    if m: rows += int(m.group(1))
        bar.value = min(rows, TOTAL)
        msg = f"{files_done}/{len(files)} 文件, {rows:,}/{TOTAL:,} ({rows/max(TOTAL,1)*100:.1f}%)"
        if _t0 and rows:
            r = rows / max(time.time()-_t0, 1); msg += f" | {r:,.0f}帖/s | 剩余{(TOTAL-rows)/max(r,1)/3600:.1f}h"
        label.value = msg; _feat_stop.wait(5)
threading.Thread(target=_w, daemon=True).start(); print("进度条已启动")

进度条已启动


In [ ]:
# 6. 产物验证 (完成后执行)
import numpy as np
# CLS
CLS = os.path.join(OUT_DIR, "gubapost_stockday_cls.parquet")
assert os.path.exists(CLS), f"CLS 不存在: {CLS} (先等提取跑完)"
df = pd.read_parquet(CLS)
cls = np.stack(df["cls"].values).astype(np.float32)
print(f"CLS: {df.shape} | cls {cls.shape} | finite={np.isfinite(cls).all()}")
print(f"  prob: [{df['prob'].min():.4f}, {df['prob'].max():.4f}] | norm={np.linalg.norm(cls,axis=1).mean():.2f}")
# Attn (dead 128 per layer)
ATN = os.path.join(OUT_DIR, "gubapost_stockday_attn.parquet")
assert os.path.exists(ATN), f"Attn 不存在: {ATN}"
da = pd.read_parquet(ATN)
attn_cols = [c for c in da.columns if c.startswith("attn_")]
a1 = np.stack(da["attn_01"].values).astype(np.float32)
a12 = np.stack(da["attn_12"].values).astype(np.float32)
print(f"\nAttn: {da.shape} | {len(attn_cols)} 层 | attn_01 {a1.shape} | finite={np.isfinite(a1).all()}")
print(f"  norm: L1={np.linalg.norm(a1,axis=1).mean():.2f} L12={np.linalg.norm(a12,axis=1).mean():.2f}")
print(f"\n日期: {df.available_date.min()} ~ {df.available_date.max()} | 股票: {df.symbol.nunique()}")
print(f"文件: CLS {os.path.getsize(CLS)/1e6:.1f}MB, Attn {os.path.getsize(ATN)/1e6:.0f}MB")